In [1]:

%pip install requests tqdm


Note: you may need to restart the kernel to use updated packages.


In [2]:

import re
from pathlib import Path
from urllib.parse import urlparse, unquote

links_file = Path("Links.txt")
assert links_file.exists(), "Put Links.txt in your notebook folder."

text = links_file.read_text(encoding="utf-8")

# Extract URLs from the exported Markdown table
urls = re.findall(r"https?://[^\s|'\"]+", text)

# Undo Markdown escaping, such as cloudfront\.net and yellow\_tripdata
urls = [re.sub(r"\\([._-])", r"\1", u.rstrip(".,;")) for u in urls]

# Keep only TLC Parquet dataset URLs
urls = [
    u for u in urls
    if "cloudfront.net/trip-data/" in u and u.endswith(".parquet")
]

urls = list(dict.fromkeys(urls))

print(f"Unique Parquet files found: {len(urls)}")
print("First 5 URLs:")
for url in urls[:5]:
    print(url)


Unique Parquet files found: 599
First 5 URLs:
https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2026-08.parquet
https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2026-07.parquet
https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2026-06.parquet
https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2026-05.parquet
https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2026-04.parquet


In [3]:

import requests
import os
import time
from concurrent.futures import ThreadPoolExecutor, as_completed
from tqdm.auto import tqdm
from pathlib import Path
from urllib.parse import urlparse, unquote

OUT_DIR = Path("TLC_Data")
OUT_DIR.mkdir(exist_ok=True)

MAX_WORKERS = 4
RETRIES = 4
CHUNK_SIZE = 1024 * 1024  # 1 MB

session = requests.Session()
session.headers.update({"User-Agent": "TLC-Data-Downloader/1.0"})

def download_file(url):
    filename = unquote(Path(urlparse(url).path).name)
    destination = OUT_DIR / filename
    temporary = OUT_DIR / (filename + ".part")

    for attempt in range(1, RETRIES + 1):
        try:
            # Skip an existing file only when its size matches the server.
            head = session.head(url, timeout=30, allow_redirects=True)
            head.raise_for_status()
            expected = int(head.headers.get("Content-Length", 0))

            if destination.exists() and expected:
                if destination.stat().st_size == expected:
                    return filename, "Already complete"

            # Restart incomplete temporary downloads safely.
            temporary.unlink(missing_ok=True)

            with session.get(url, stream=True, timeout=(30, 180)) as response:
                response.raise_for_status()
                response_size = int(response.headers.get("Content-Length", 0))
                with open(temporary, "wb") as f:
                    for chunk in response.iter_content(CHUNK_SIZE):
                        if chunk:
                            f.write(chunk)

            actual = temporary.stat().st_size
            if response_size and actual != response_size:
                raise IOError(
                    f"Incomplete download: expected {response_size}, got {actual}"
                )

            temporary.replace(destination)
            return filename, f"Downloaded ({actual / 1024**2:.1f} MB)"

        except Exception as exc:
            temporary.unlink(missing_ok=True)
            if attempt == RETRIES:
                return filename, f"FAILED: {exc}"
            time.sleep(2 * attempt)

results = []

with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
    futures = {executor.submit(download_file, url): url for url in urls}

    for future in tqdm(as_completed(futures), total=len(futures),
                       desc="Downloading TLC files"):
        results.append(future.result())

# Save a report so failed downloads can be identified later.
import pandas as pd

report = pd.DataFrame(results, columns=["filename", "status"])
report.to_csv(OUT_DIR / "download_report.csv", index=False)

print("\nDownload summary:")
print(report["status"].str.startswith("FAILED").value_counts())
display(report[report["status"].str.startswith("FAILED")])



Download summary:
status
False    347
True     252
Name: count, dtype: int64


,filename,status
347,green_tripdata_2015-05.parquet,FAILED: 403 Client Error: Forbidden for url: h...
348,green_tripdata_2015-04.parquet,FAILED: 403 Client Error: Forbidden for url: h...
349,green_tripdata_2015-03.parquet,FAILED: 403 Client Error: Forbidden for url: h...
350,green_tripdata_2015-02.parquet,FAILED: 403 Client Error: Forbidden for url: h...
351,green_tripdata_2015-01.parquet,FAILED: 403 Client Error: Forbidden for url: h...
...,...,...
594,fhv_tripdata_2015-05.parquet,FAILED: 403 Client Error: Forbidden for url: h...
595,fhv_tripdata_2015-03.parquet,FAILED: 403 Client Error: Forbidden for url: h...
596,fhv_tripdata_2015-04.parquet,FAILED: 403 Client Error: Forbidden for url: h...
597,fhv_tripdata_2015-02.parquet,FAILED: 403 Client Error: Forbidden for url: h...


In [4]:

files = list(OUT_DIR.glob("*.parquet"))

total_bytes = sum(f.stat().st_size for f in files)

print(f"Parquet files present: {len(files)}")
print(f"Total size: {total_bytes / 1024**3:.2f} GB")
print(f"Output folder: {OUT_DIR.resolve()}")

display(report.tail(10))


Parquet files present: 347
Total size: 31.04 GB
Output folder: C:\Users\himni\Downloads\ilovepdf_pages-to-jpg (1)\BDA Data\TLC_Data


,filename,status
589,fhv_tripdata_2015-10.parquet,FAILED: 403 Client Error: Forbidden for url: h...
590,fhv_tripdata_2015-09.parquet,FAILED: 403 Client Error: Forbidden for url: h...
591,fhv_tripdata_2015-07.parquet,FAILED: 403 Client Error: Forbidden for url: h...
592,fhv_tripdata_2015-08.parquet,FAILED: 403 Client Error: Forbidden for url: h...
593,fhv_tripdata_2015-06.parquet,FAILED: 403 Client Error: Forbidden for url: h...
594,fhv_tripdata_2015-05.parquet,FAILED: 403 Client Error: Forbidden for url: h...
595,fhv_tripdata_2015-03.parquet,FAILED: 403 Client Error: Forbidden for url: h...
596,fhv_tripdata_2015-04.parquet,FAILED: 403 Client Error: Forbidden for url: h...
597,fhv_tripdata_2015-02.parquet,FAILED: 403 Client Error: Forbidden for url: h...
598,fhv_tripdata_2015-01.parquet,FAILED: 403 Client Error: Forbidden for url: h...
